<a href="https://colab.research.google.com/github/erlinsari/EmosiKu/blob/main/EmosiKu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EmosiKu: Klasifikasi Indikasi Gangguan Kesehatan Mental pada Teks Media Sosial Berbahasa Indonesia Menggunakan Model IndoBERT

**Sistem:** Deteksi Depresi dan Kecemasan berbasis NLP (Text Classification)
**Model Utama:** IndoBERT


In [3]:
!pip install transformers datasets Sastrawi scikit-learn pandas numpy torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 7.6 MB/s eta 0:00:00


## 1. Import Library & Koneksi Google Drive

In [5]:
import pandas as pd
import numpy as np
import re
import torch
import os
from google.colab import drive
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Hubungkan Google Colab dengan Google Drive Anda
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Load Dataset dari Google Drive
Dataset dibaca langsung dari folder yang ada di Google Drive Anda. Pastikan Anda menyesuaikan `path_folder` di bawah ini dengan lokasi folder data Anda di Drive.

In [6]:
path_folder = '/content/drive/MyDrive/EmosiKu/DATASET/'

try:
    train_df = pd.read_csv(os.path.join(path_folder, 'datd_train.csv'))
    test_df = pd.read_csv(os.path.join(path_folder, 'datd_test.csv'))
    print("Dataset berhasil dimuat!")
    display(train_df.head())
except Exception as e:
    print(f"Error memuat dataset: {e}")

Dataset berhasil dimuat!


,text,label
0,"oh pantesan tadi pada rame, ternyata monek mau...",0
1,"Semakin bertambah usia, semakin cemas hidup.",0
2,gelisah bgt astaga,1
3,Udah jangan terlalu cemas sikapku tak berubah ...,0
4,Giliran Aldebaran diambang kematian...Semua ba...,0


## 3. Preprocessing Data

In [7]:
factory_stopword = StopWordRemoverFactory()
stopword = factory_stopword.create_stop_word_remover()

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.lower()
    text = stopword.remove(text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train_df['clean_text'] = train_df['text'].apply(clean_text)
test_df['clean_text'] = test_df['text'].apply(clean_text)

display(train_df[['text', 'clean_text', 'label']].head())

,text,clean_text,label
0,"oh pantesan tadi pada rame, ternyata monek mau...",pantesan tadi rame ternyata monek mau cb juni ...,0
1,"Semakin bertambah usia, semakin cemas hidup.",semakin bertambah usia semakin cemas hidup,0
2,gelisah bgt astaga,gelisah bgt astaga,1
3,Udah jangan terlalu cemas sikapku tak berubah ...,udah jangan terlalu cemas sikapku tak berubah ...,0
4,Giliran Aldebaran diambang kematian...Semua ba...,giliran aldebaran diambang kematiansemua baru ...,0


## 4. Tokenisasi dengan IndoBERT

In [8]:
model_name = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_encodings = tokenizer(train_df['clean_text'].tolist(), truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(test_df['clean_text'].tolist(), truncation=True, padding=True, max_length=128)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [9]:
class MentalHealthDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = MentalHealthDataset(train_encodings, train_df['label'].tolist())
test_dataset = MentalHealthDataset(test_encodings, test_df['label'].tolist())

## 5. Fine-Tuning Model IndoBERT

In [14]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall}

# --- BAGIAN INI YANG KITA OPTIMASI ---
training_args = TrainingArguments(
    output_dir='./EmosiKu_Model',
    num_train_epochs=8,              # Diubah menjadi 8 agar akurasi lebih tinggi
    learning_rate=2e-5,              # Ditambahkan agar belajar lebih teliti
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",      # Ditambahkan agar model fokus mencari F1-Score terbaik
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


## 6. Training & Evaluasi Model

In [15]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.581011,0.414597,0.818182,0.618321,0.801980,0.503106
2,0.479426,0.382793,0.827273,0.719764,0.685393,0.757764
3,0.392286,0.351865,0.843636,0.713333,0.769784,0.664596
4,0.251309,0.514122,0.825455,0.743316,0.652582,0.863354
5,0.168859,0.614698,0.840000,0.704698,0.766423,0.652174
6,0.121400,0.683411,0.829091,0.709877,0.705521,0.714286
7,0.110094,0.812485,0.821818,0.699387,0.690909,0.708075
8,0.071812,0.843775,0.823636,0.695925,0.702532,0.689441


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=1104, training_loss=0.2699853232521834, metrics={'train_runtime': 566.9789, 'train_samples_per_second': 31.056, 'train_steps_per_second': 1.947, 'total_flos': 660544415591520.0, 'train_loss': 0.2699853232521834, 'epoch': 8.0})

In [16]:
eval_results = trainer.evaluate()
print(f"Hasil Evaluasi: {eval_results}")

Hasil Evaluasi: {'eval_loss': 0.5133959650993347, 'eval_accuracy': 0.8254545454545454, 'eval_f1': 0.7433155080213903, 'eval_precision': 0.6525821596244131, 'eval_recall': 0.8633540372670807, 'eval_runtime': 1.8622, 'eval_samples_per_second': 295.352, 'eval_steps_per_second': 4.833, 'epoch': 8.0}


In [17]:
# Simpan model terbaik ke folder Google Drive agar tidak hilang saat Colab ditutup
model_save_path = '/content/drive/MyDrive/EmosiKu_Model/best_model'
os.makedirs(model_save_path, exist_ok=True)
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)
print(f"Model berhasil disimpan secara permanen di {model_save_path}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model berhasil disimpan secara permanen di /content/drive/MyDrive/EmosiKu_Model/best_model
